# Stage 3.1 — the baseline run

The reference every later run is compared against (B10). The suite is
executed once at full `n` (seeds at 8, follow-ups at 4, D21). Every execution
is checked structurally and scored point-wise, and the author adopts **one
execution per case** as that case's baseline reply, in `cases/baseline.json`.

This notebook shows, per case, what the run did (structural outcomes, scores,
cost, latency), shows a failure rather than hiding it (§3.5), puts the
top-scoring reply in front of the author for review, and checks that the
adoption is complete.

**The parameters decide which run it describes.** Pointed at a test baseline
(any file outside `cases/`), it says so in every output that matters.

In [1]:
import sys, pathlib, textwrap
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, comparison as cmp
from src.cases import load_cases
from src.scoring import first_reply, _turn_counts
from src import db

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

# --- parameters -------------------------------------------------------------
# The runs the baseline is drawn from. The real baseline is ONE run at full n:
# B10, run 1a872c09 (2026-09-25/26, 31 cases, 176 executions; resumed once
# after a harness session lapse, D95).
BASELINE_RUN_IDS = ["1a872c09-7070-4789-a806-904969f9185d"]
# cases/baseline.json is the author's adoption; anything else is a test file.
# The proposal for this run; the author adopts it into cases/baseline.json.
BASELINE_FILE = cmp.RUNS_DIR / "baseline_proposal_1a872c09.json"
# Write the proposal to BASELINE_FILE. Refused for the adopted file: the author
# adopts by instruction, not by a notebook cell.
WRITE_PROPOSAL = True
# ---------------------------------------------------------------------------

TEST = cmp.is_test_baseline(BASELINE_FILE)
print("TEST BASELINE - a rehearsal, not the adopted baseline" if TEST else "ADOPTED BASELINE (cases/baseline.json)")
for rid in BASELINE_RUN_IDS:
    print()
    print(survey.headline(rid))
cases = {c.case_id: c for c in load_cases()}
print()
print(f"suite: {len(cases)} cases")

TEST BASELINE - a rehearsal, not the adopted baseline



run 1a872c09  started 2026-09-25T22:27:16
  notes         B10 baseline — overnight, started by tools/overnight_baseline.py
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.8 @ b954dbe9b545
  suite         a1d6a7f96d8d
  executions    176  (+0 superseded)
  failed        0
  scored        176, mean 9.26
  cost          $4.5164



suite: 31 cases


## 1. The run records its configuration

Every version field must be set, or the comparison in stage 3.2 cannot show
that nothing changed between the runs. Where the baseline spans several runs
(only in a rehearsal), fields that differ between them are listed, because a
real baseline cannot have them.

In [2]:
VERSION_FIELDS = ["model_version", "model_released_on", "prompt_version", "tool_version",
                  "suite_version", "api_version", "api_commit", "app_version", "git_sha",
                  "simulator_model", "simulator_prompt_version", "judge_model", "judge_prompt_version"]
runs = pd.DataFrame([db.admin().table("study_runs").select("*").eq("id", r).execute().data[0]
                     for r in BASELINE_RUN_IDS]).set_index("id")
versions = runs[VERSION_FIELDS].T
versions.columns = [c[:8] for c in versions.columns]
missing = [f for f in VERSION_FIELDS if runs[f].isna().any()]
differ = [f for f in VERSION_FIELDS if runs[f].nunique(dropna=False) > 1]
print(f"version fields unset: {missing or 'none'}")
print(f"fields that differ between the baseline's runs: {differ or 'none'}")
versions

version fields unset: none
fields that differ between the baseline's runs: none


,1a872c09
model_version,gemini-3.5-flash
model_released_on,2026-05-19
prompt_version,382dc09207a2
tool_version,2e6f598f22a1
suite_version,a1d6a7f96d8db36665b1d14088c4c5937bc2c77f5aabc6935c6df3785bb3ab9f
api_version,0.1.8
api_commit,b954dbe9b545
app_version,0.1.7
git_sha,b954dbe9b545d99106afba68c2c2ec1f47bf4c55
simulator_model,gemini-3.5-flash


## 2. What each case did

One row per case: structural outcomes across its repetitions, the point-wise
scores, cost and latency.

**`later turn acted (not judged)`**: how many of the case's executions did
real work *after* the first turn, i.e. a later turn called a tool (drafted a
plan, moved a stop). The judge scores the first reply only (D34, D85), so
for these executions the score describes the opening reply and **not** that
later work. The structural checks do cover it. A later turn that only says
"no thanks" is not counted, because nothing was left unjudged. Non-zero only
for the two multi-step seeds, whose second turn is the point of the case.

In [3]:
frame = pd.concat([survey.run_frame(r).assign(run=r[:8]) for r in BASELINE_RUN_IDS], ignore_index=True)
executions = cmp._executions(BASELINE_RUN_IDS)
by_exec = {e["id"]: e for e in executions}
acted_later = {e["case_id"]: 0 for e in executions}
for e in executions:
    acted_later[e["case_id"]] += _turn_counts(e)["later_acting_turns"] > 0

per_case = frame.groupby("case").agg(
    function=("function", "first"), provenance=("provenance", "first"),
    n=("repetition", "count"),
    structural_none=("structural", lambda s: int((s == "none").sum())),
    structural_detected=("structural", lambda s: int((s == "detected").sum())),
    failed=("blame", lambda s: int((s != "-").sum())),
    score_mean=("score", "mean"), score_min=("score", "min"), score_max=("score", "max"),
    cost_usd=("cost_usd", "sum"), latency_s=("latency_ms", lambda s: round(s.mean() / 1000, 1)),
)
per_case["later turn acted (not judged)"] = per_case.index.map(acted_later)
not_run = sorted(set(cases) - set(per_case.index))
print(f"{len(frame)} executions over {len(per_case)} cases; cases never executed: {not_run or 'none'}")
per_case.round(2)

176 executions over 31 cases; cases never executed: none


,function,provenance,n,structural_none,structural_detected,failed,score_mean,score_min,score_max,cost_usd,latency_s,later turn acted (not judged)
case,,,,,,,,,,,,
advisory_packing_list,advisory,seeded,8,8,0,0,10.00,10,10,0.31,46.5,0
advisory_packing_list__inv1_paraphrase,advisory,metamorphic,4,4,0,0,9.75,9,10,0.14,48.7,0
advisory_packing_list__inv3_irrelevant_detail,advisory,metamorphic,4,4,0,0,9.50,9,10,0.16,48.9,0
advisory_sintra_full_or_half_day,advisory,seeded,8,8,0,0,10.00,10,10,0.16,20.7,0
advisory_sintra_full_or_half_day__inv1_paraphrase,advisory,metamorphic,4,4,0,0,10.00,10,10,0.08,16.4,0
advisory_sintra_full_or_half_day__inv4_german,advisory,metamorphic,4,4,0,0,10.00,10,10,0.08,20.0,0
factual_qa_alfama_history,factual_qa,seeded,8,8,0,0,10.00,10,10,0.14,19.7,0
factual_qa_alfama_history__dir1_the_earthquake,factual_qa,metamorphic,4,4,0,0,10.00,10,10,0.07,18.4,0
factual_qa_alfama_history__inv1_paraphrase,factual_qa,metamorphic,4,4,0,0,10.00,10,10,0.07,19.7,0


## 3. Two failures, shown

§3.5: one worked example of a failure is worth more than three of successes.
Two are shown, because this run failed in two different ways:

1. **The lowest-scoring structural failure**, picked automatically. In this
   run that is the surf case, where a tool failed underneath the agent.
2. **A scheduling failure in the tram case** (`scheduling_afternoon_then_tram`),
   picked by what it shows: the one repetition where the agent's own two
   actions contradict each other.

A structural failure is in what the agent **did**, so that is what is shown,
in the order the checker reads it: what the case asserts, then turn by turn
what the member asked, which tools ran, what the draft held and what changed
on the live itinerary, then the checker's findings. The reply comes last,
because the judge's view of this execution is not where it failed.


In [4]:
from src.execution import load
from src.checker import _slot_entries

def show_execution(ex_row, case):
    e = load(ex_row)
    checks = case.declared_constraints.checks.model_dump(exclude_none=True)
    print("THE CASE ASSERTS")
    for key in ("expected_tool_sequence",):
        print(f"  {key}: {getattr(case, key)}")
    for key, value in checks.items():
        print(f"  {key}: {value}")
    ids = {v: k for k, v in e.blocks.items()}
    for turn in e.turns:
        t = turn["turn_index"]
        print()
        print(f"TURN {t}  member: {(turn.get('user_turn') or '')[:150]!r}")
        for call in (c for c in e.trace if c["turn_index"] == t):
            err = f"  ERROR {str(call['result_error'])[:80]}" if call.get("result_error") else ""
            print(f"  tool  {call['tool']}{err}")
        for draft, dt in zip(e.drafts, e.draft_turns):
            if dt != t:
                continue
            print("  draft proposed:")
            for s in _slot_entries(draft):
                b = s.get("block") or {}
                ref = b.get("reuse_block_id") or ""
                saved = next((title for bid, title in ids.items() if ref and bid.startswith(ref)), None)
                tag = f"   <- reuses saved stop '{saved}'" if saved else ""
                print(f"    {s['_day']} {s.get('start_time')}-{s.get('end_time')}  {b.get('title')}{tag}")
    live = [m for m in e.mutations if m["table_name"] == "itinerary_slots"]
    print()
    print("LIVE ITINERARY CHANGES")
    for m in live or [None]:
        if m is None:
            print("  none")
            break
        pr = m.get("projection") or {}
        print(f"  {m['op']} {ids.get(pr.get('trip_block_id'), pr.get('trip_block_id'))}: "
              f"{pr.get('day_date')} {str(pr.get('start_time'))[:5]}-{str(pr.get('end_time'))[:5]}")
    print()
    print(f"CHECKER: {ex_row.get('structural_outcome')}")
    for f in ex_row.get("structural_findings") or []:
        print("  finding:", f)
    request, reply = first_reply(ex_row)
    print()
    print("FIRST REPLY (what the judge scored)")
    print(textwrap.fill(reply[:600], 110))

def show_pick(row):
    ex = next(e for e in executions if e["case_id"] == row["case"] and e["repetition"] == row["repetition"]
              and e["run_id"][:8] == row["run"])
    print(f"{row['case']} rep {row['repetition']} (run {row['run']}), judge score {row['score']}")
    print()
    show_execution(ex, cases[row["case"]])

bad = frame[frame["structural"] == "detected"]

# 1. The lowest-scoring structural failure (else the lowest score overall).
pick = (bad if len(bad) else frame.dropna(subset=["score"])).sort_values("score").head(1)
print("=" * 110)
print("FAILURE 1 - the lowest-scoring structural failure")
print("=" * 110)
if pick.empty:
    print("no execution to show")
else:
    show_pick(pick.iloc[0])

# 2. The tram case: the repetition where the move contradicts the agent's own
#    draft (its finding names the draft), else its first structural failure.
tram = bad[bad["case"] == "scheduling_afternoon_then_tram"]
if len(tram):
    by_id = {(e["case_id"], e["repetition"], e["run_id"][:8]): e for e in executions}
    overlap = tram[tram.apply(lambda r: any("before the draft" in f for f in
                   by_id[(r["case"], r["repetition"], r["run"])].get("structural_findings") or []), axis=1)]
    chosen = (overlap if len(overlap) else tram).sort_values("repetition").iloc[0]
    print()
    print("=" * 110)
    print("FAILURE 2 - the tram case")
    print("=" * 110)
    show_pick(chosen)


FAILURE 1 - the lowest-scoring structural failure
factual_qa_lisbon_surf_conditions rep 2 (run 1a872c09), judge score 1



THE CASE ASSERTS
  expected_tool_sequence: []
  tool_counts: {'get_region_climate': {'min': 0, 'max': 1}, 'get_weather': {'min': 0, 'max': 1}, 'get_climate_normals': {'min': 1, 'max': 2}}
  tools_any_of: ['get_climate_normals']
  tool_errors: corrected
  turn_errors: none
  mutation_counts: {'trip_blocks.INSERT': 0, 'itinerary_slots.INSERT': 0, 'itinerary_slots.UPDATE': 0, 'ai_suggestions.INSERT': 0}
  trip_after: {'blocks': 'fixture', 'slots': 'fixture'}

TURN 1  member: '@Tonic what are the waves and wind like around Lisbon in November? thinking of bringing a board'
  tool  get_climate_normals  ERROR Climate data unavailable: The operation was aborted due to timeout
  tool  get_region_climate
  tool  read_wikipedia
  tool  search_places

TURN 2  member: "No thanks on the guide. I just need the figures: what's the average wave height in meters and typical wind speed in knots for November?"
  tool  get_climate_normals

LIVE ITINERARY CHANGES
  none

CHECKER: detected
  finding: called 

THE CASE ASSERTS
  expected_tool_sequence: ['search_places', 'propose_plan', 'move_slot']
  tool_counts: {'search_places': {'min': 1, 'max': 4}, 'propose_plan': {'min': 1, 'max': 2}, 'move_slot': {'min': 1, 'max': 2}, 'get_weather': {'min': 0, 'max': 1}}
  tool_errors: corrected
  turn_errors: none
  places_surfaced: {'min': 1}
  places_near: {'anchor': 'Tram 28 ride through Alfama', 'within_km': 3.0}
  slot_days: ['2026-11-06']
  slot_count: {'min': 2, 'max': 3}
  draft_slot_starts_between: ('14:00', '18:00')
  slot_moved: {'block': 'Tram 28 ride through Alfama', 'day': '2026-11-06', 'starts_between': ('15:00', '19:30'), 'duration_minutes': {'min': 45, 'max': 90}, 'clear_of_other_slots': True, 'after_latest_draft': True}
  edits: {'move': {'min': 1, 'max': 2}, 'unschedule': 0, 'schedule': 0}
  mutation_counts: {'trip_blocks.INSERT': 0, 'itinerary_slots.INSERT': 0, 'itinerary_slots.UPDATE': {'min': 1, 'max': 2}, 'ai_suggestions.INSERT': {'min': 1, 'max': 2}}
  mutation_changed_keys: {'

**Reading failure 1: an unreliable tool, a sensible workaround, a correct detection.**

`factual_qa_lisbon_surf_conditions` asks for November wave height and wind, and
the case permits only the climate tools. In this repetition the agent did call
the right one first: `get_climate_normals`. **The tool failed**: "Climate data
unavailable: The operation was aborted due to timeout". The API gives an
Open-Meteo request 8 seconds (`OPEN_METEO_TIMEOUT_MS` in
`apps/api/src/routes/ai/lib/weather.ts`), and the reanalysis query behind
climate normals did not answer in time. It timed out in **2 of the case's 8
repetitions** (rep 2 and rep 4); the other calls succeeded.

The agent then worked around the error rather than failing: it fell back to
`get_region_climate` and, when that gave no marine figures, to `read_wikipedia`
(and here also `search_places`). **As behaviour, that is reasonable**: a member
with a failed tool is better served by an alternative source than by nothing.
It still produced a reply without the figures the case asks for, and the judge
scored it 1/10. In turn 2 the member asked for the numbers outright, and a
second `get_climate_normals` call succeeded; the judge scores turn 1 only (D34).

**The suite is right to flag it.** The case fixes which tools answer a
climate question, so a route through Wikipedia is a changed route, whatever
prompted it, and that is what the finding says. What the finding also reveals
is the more useful fact: **`get_climate_normals` is unreliable under an
8-second budget**, and a quarter of this case's repetitions depended on luck
with an upstream service. Two readings follow, and they point at different
fixes: the agent recovered sensibly (so the case could permit the fallback, as
D68 did for the two climate tools), or the tool needs a longer budget or a
retry (a change to the application, outside the study's scope until after the
round). The third detected repetition (rep 8) called `search_places` without any
tool error, so not every route change here was forced by the timeout.

**Reading failure 2: two correct actions that contradict each other.**

`scheduling_afternoon_then_tram` has two steps. The member asks Tonic to fill
Friday afternoon (turn 1, a `/plan` draft), then asks to move the tram, already
scheduled that morning, "to the very end of this afternoon" (turn 2, a live
move). In this repetition both steps look right on their own:

- The draft is fine: three Alfama stops from 14:30, the last one ending at 18:00.
- The move is fine as a move: the tram now runs 17:30-18:30 on Friday.

But the agent placed the tram by reading the **live itinerary**
(`get_itinerary`), and the draft is not on it yet: it is a proposal the group
accepts slot by slot. So the agent overlooked its own draft. Once the group
accepts it, the tram overlaps the viewpoint (17:15-18:00) by 30 minutes, and
"the very end of the afternoon" is no longer the end. The judge scored the reply
10/10, because the reply is about the draft and reads well; only the
`after_latest_draft` check, which reads times rather than prose, catches it.

**The other six failures of this case are a different pattern.** In reps 1, 2,
4, 5, 6 and 7 the agent answered the move by re-drafting the afternoon and put
already-scheduled stops into the new draft (the saved lunch at 12:30 and/or the
tram itself). A draft is a set of offers to *add* stops, so accepting it would
schedule those stops a second time. The checker sees this indirectly, as a
draft of 4-5 slots where the member asked for "two or three stops", some
starting outside the afternoon window. Only rep 3 handled both steps cleanly.


## 4. The proposed baseline, for review

Per case, the execution `comparison.propose_baseline` would adopt: structurally
clean first, then the highest judge score, then the earliest repetition. Its
first reply is printed, because that is the text later runs are judged against.
**The author reads these and adopts, or picks another execution.** Adoption is
the author's act (B10).

Where the judge scored the shown reply below 10, its reasoning is printed under
it (`judge:`, persisted in `study_scores`, D71): a 10 needs no explanation, a
deduction names the criterion the reply missed.

In [5]:
proposal = cmp.propose_baseline(BASELINE_RUN_IDS)
unclean = proposal[proposal["structural"] != "none"]
print(f"proposal covers {len(proposal)} of {len(cases)} cases; "
      f"cases whose best execution is not structurally clean: {list(unclean.index) or 'none'}")
proposal[["run_id", "repetition", "structural", "score"]].assign(run_id=lambda f: f["run_id"].str[:8])

proposal covers 31 of 31 cases; cases whose best execution is not structurally clean: none


,run_id,repetition,structural,score
case,,,,
advisory_packing_list,1a872c09,1,none,10
advisory_packing_list__inv1_paraphrase,1a872c09,2,none,10
advisory_packing_list__inv3_irrelevant_detail,1a872c09,1,none,10
advisory_sintra_full_or_half_day,1a872c09,1,none,10
advisory_sintra_full_or_half_day__inv1_paraphrase,1a872c09,1,none,10
advisory_sintra_full_or_half_day__inv4_german,1a872c09,1,none,10
factual_qa_alfama_history,1a872c09,1,none,10
factual_qa_alfama_history__dir1_the_earthquake,1a872c09,1,none,10
factual_qa_alfama_history__inv1_paraphrase,1a872c09,1,none,10


In [6]:
rationale_of = {
    r["execution_id"]: r["rationale"]
    for r in db.admin().table("study_scores").select("execution_id, rationale")
    .in_("run_id", BASELINE_RUN_IDS).execute().data
}

for case_id, row in proposal.iterrows():
    request, reply = first_reply(by_exec[row["execution_id"]])
    print("=" * 110)
    print(f"{case_id}   [score {row['score']}, rep {row['repetition']}, run {row['run_id'][:8]}]")
    print("request:", request)
    print(textwrap.fill(reply[:700] + (" ..." if len(reply) > 700 else ""), 110))
    if row["score"] is not None and row["score"] < 10:
        print()
        print("judge:")
        print(textwrap.fill(rationale_of.get(row["execution_id"]) or "(no rationale)", 110,
                            initial_indent="    ", subsequent_indent="    "))

advisory_packing_list   [score 10, rep 1, run 1a872c09]
request: @Tonic what should the three of us pack for this trip?
For your trip in early November (Nov 6–9), Lisbon and Sintra transition into cooler, wetter autumn weather.
November is historically the wettest month of the year in Lisbon, with average highs around 18°C (64°F) and
lows dropping to 11°C (53°F), while Sintra's forested hills will feel noticeably cooler, damp, and breezy.
Here is what the three of us should pack:  *   **Footwear with grip:** Lisbon's iconic cobblestones (*calçada
portuguesa*) get incredibly slick when wet, and Sintra involves steep walking. Bring comfortable, broken-in
sneakers or boots with excellent traction. *   **Rain protection & layers:** A lightweight, windproof, and
waterproof jacket is essential. Layering is key—br ...
advisory_packing_list__inv1_paraphrase   [score 10, rep 2, run 1a872c09]
request: @Tonic we're trying to work out what to bring — any thoughts for the three of us?
For your trip

advisory_packing_list__inv3_irrelevant_detail   [score 10, rep 1, run 1a872c09]
request: @Tonic what should the three of us pack for this trip? we just bought a huge new suitcase so space isn't really a problem
Since your trip is in early November (Nov 6–9), you'll be visiting during one of Lisbon's wettest and most
transitional times of year. Typical November temperatures hover around a mild high of 18°C (64°F) and drop to
a cool 12°C (54°F) at night, with Sintra's forested hills feeling noticeably colder and damper.   Since you
have plenty of suitcase space, here is what the three of you should pack:  *   **Footwear with serious grip:**
Lisbon's iconic limestone cobblestones (*calçada portuguesa*) are notoriously slippery, especially when wet.
Leave heels or smooth-soled shoes behind and pack broken-in sneakers or boots with excellent traction for
walking up and down the steep hills. *   **Rain g ...
advisory_sintra_full_or_half_day   [score 10, rep 1, run 1a872c09]
request: @Tonic i

advisory_sintra_full_or_half_day__inv1_paraphrase   [score 10, rep 1, run 1a872c09]
request: @Tonic do you reckon Sintra needs a whole day, or would half of one do?
Sintra definitely warrants a **full day** (and even then, you'll only scratch the surface!).   Because it is
set high up in the hills, getting between the major sights—like the Pena Palace (which you already have saved
and scheduled for Day 3), the Moorish Castle, and Quinta da Regaleira—takes time due to winding mountain
roads, one-way traffic systems, and tourist crowds. A half-day trip usually leaves people feeling incredibly
rushed, often only allowing time to see one palace before having to head straight back.   With a full day, you
can comfortably explore the stunning Pena Palace gardens and interior in the morning, grab lunch in the
historic town center, and fit in a second spectacula ...
advisory_sintra_full_or_half_day__inv4_german   [score 10, rep 1, run 1a872c09]
request: @Tonic lohnt sich Sintra für einen ganzen

factual_qa_alfama_history   [score 10, rep 1, run 1a872c09]
request: @Tonic what's the history of Alfama, briefly?
Alfama is Lisbon's oldest neighborhood, and during Moorish rule, it actually made up the entire city before it
expanded westward. Because of its dense, rocky foundations, it miraculously survived the devastating 1755
Lisbon earthquake, leaving its medieval labyrinth of narrow streets, alleys, and small squares beautifully
intact. Historically a home for fishermen and the working class, it has evolved today into a vibrant area
famous for its traditional Fado houses and stunning viewpoints.
factual_qa_alfama_history__dir1_the_earthquake   [score 10, rep 1, run 1a872c09]
request: @Tonic what happened to Alfama in the 1755 earthquake?
Remarkably, Alfama largely survived the devastating 1755 Lisbon earthquake! Because it is built on a dense,
solid bedrock of basalt and clay high up on the hill, the neighborhood escaped the worst of the shaking and
the subsequent tsunami.   This

factual_qa_alfama_history__inv1_paraphrase   [score 10, rep 1, run 1a872c09]
request: @Tonic give me the short version of how Alfama came to be what it is
During Moorish rule, Alfama made up the entire city of Lisbon before it expanded westward. It survived the
devastating 1755 earthquake, which is why it remains a historic, labyrinthine maze of narrow streets and small
squares that historically became home to fishermen and the city's working class.
factual_qa_lisbon_forecast_next_days   [score 9, rep 1, run 1a872c09]
request: @Tonic what's the weather doing in Lisbon over the next couple of days?
It is looking very warm but overcast in Lisbon over the next few days!   * **Today (Friday, Sep 25):**
Overcast with a high of 30°C (17°C at night) and no rain. * **Tomorrow (Saturday, Sep 26):** Still overcast
but reaching a hot 33°C (20°C at night). * **Sunday (Sep 27):** Overcast, around 30°C with a tiny 3% chance of
rain. * **Monday (Sep 28):** Slightly cooler at 26°C, remaining overcast.

factual_qa_lisbon_november_weather   [score 9, rep 5, run 1a872c09]
request: @Tonic what's the weather usually like in Lisbon in November?
In November, Lisbon is generally mild but it is historically the wettest month of the year. You can expect
average daytime highs of around 18.1°C (65°F) and nighttime lows of about 11.8°C (53°F).   It rains on average
about 10 days of the month, bringing around 134 mm of precipitation, so packing a light rain jacket or
umbrella is a smart move. However, you'll still get plenty of bright intervals, with the sun shining for about
54% of the daylight hours.

judge:
    The reply provides highly accurate climate figures for Lisbon in November that perfectly match the
    reference data (18.1°C high, 11.8°C low, 134 mm of rain). It correctly identifies November as historically
    the wettest month and frames the information as long-run historical averages rather than a specific
    forecast. It only slightly misses explicitly naming the specific source 

... (7 outputs left out)

venue_lookup_seafood_lisbon__dir1_and_a_hotel   [score 10, rep 1, run 1a872c09]
request: /find good seafood places near Cais do Sodré, nothing too pricey — and somewhere to stay nearby too
Here are a few great, budget-friendly seafood spots and places to stay right around Cais do Sodré:  **Seafood
Spots (Casual & Great Value)** *   **Sol e Pesca**: A quirky, iconic spot in a former fishing tackle shop
serving high-quality Portuguese canned seafood, cold beers, and simple bites. *   **Uma Marisqueira**: Famous
for its legendary, massive pots of seafood rice (*arroz de marisco*)—it's no-frills, incredibly filling, and
very reasonably priced. *   **Quatro Estações**: A local, traditional Portuguese tavern serving fresh grilled
fish and classic seafood plates without the tourist markup.  **Places to Stay Nearby** *   **Sunset
Destination Hostel**: Located right inside the Cais do  ...


venue_lookup_seafood_lisbon__dir4_not_a_meal   [score 10, rep 2, run 1a872c09]
request: /find where can we get a drink by the river near Cais do Sodre?
Here are three great spots right by the river near Cais do Sodré, perfect for matching the relaxed vibe of
your trip:  *   **Quiosque Ribeira das Naus**: A casual outdoor kiosk right on the riverfront path with deck
chairs, perfect for a sunset beer or cocktail right by the water. *   **TEJÁ HOUSE**: A stylish, slightly
upscale spot right on the docks of Cais do Sodré with fantastic river views and great drinks. *   **Pensão
Amor**: Just a short walk up from the river on the famous Pink Street, this quirky, historic former brothel is
a must-visit for creative cocktails and a lively atmosphere.  Would you like me to pre-schedule one of these
for drinks on your first evening after your Time Ou ...
venue_lookup_seafood_lisbon__inv2_typos   [score 10, rep 1, run 1a872c09]
request: /find god seafod palces near cais do sodre, nothin too price

venue_lookup_sintra_lunch   [score 9, rep 1, run 1a872c09]
request: /find where should we eat lunch on the Sintra day? somewhere near the palace, proper local food
Here are three great, highly-rated local spots not far from the palace area for a proper Portuguese lunch on
your Sintra day (assuming you'll head down towards the historic center after visiting Pena Palace):  *
**Cantinho da Vila** – A cozy, traditional spot right in the historic center serving fantastic local petiscos
(Portuguese tapas) and classic dishes. * **Tasca d'Ana** – A warm, authentic tavern known for its friendly
service and comforting, home-cooked Portuguese favorites. * **Tasco do Strauss** – A highly-rated, relaxed
local gem offering excellent traditional plates and a great atmosphere.  Would you like me to pre-schedule one
of these for lunchtime on Day 3 (Sunday, Nov 8) right ...

judge:
    The assistant successfully identified the correct day (Sunday, Nov 8) for the Sintra trip and suggested
    three local

venue_lookup_sintra_lunch__dir2_lisbon_day   [score 10, rep 1, run 1a872c09]
request: /find where should we eat lunch on the last day? near the hotel in Baixa, proper local food
Here are three great, authentic spots for a proper local lunch on your last day (Monday, Nov 9th), all located
right in or very close to Baixa near your hotel:  *   **Taberna da Baixa**: A cozy, traditional tavern right
in the heart of Baixa serving classic Portuguese dishes like codfish and octopus in a warm, rustic setting. *
**Restaurante "Leitaria A Camponeza"**: A historic, highly-rated spot tucked away on Rua dos Sapateiros,
famous for its authentic local feel, great steaks, and traditional seafood. *   **Tasca Lisboeta**: Just a
short walk up towards the edge of Chiado, this is a fantastic, highly-rated family-run spot serving
incredible, comforting Portuguese plates.  Would you lik ...


## 5. Adoption

A test baseline may be written from the proposal here. The adopted one is
written only on the author's instruction, and this cell refuses to write it.

In [7]:
if WRITE_PROPOSAL and TEST:
    cmp.write_baseline(proposal, BASELINE_FILE, by="agent (pipeline rehearsal)",
                       note="TEST baseline - proposal written by 03_baseline, not an adoption")
    print(f"wrote {BASELINE_FILE.relative_to(cmp.RUNS_DIR.parent)}")
elif WRITE_PROPOSAL:
    print("refused: cases/baseline.json is the author's adoption and is not written from a notebook")

baseline = cmp.load_baseline(BASELINE_FILE) if BASELINE_FILE.exists() else {"cases": {}}
entries = baseline.get("cases", {})
gaps = cmp.baseline_gaps(baseline)
differs = [c for c, e in entries.items() if c in proposal.index and e["execution_id"] != proposal.at[c, "execution_id"]]
print(f"baseline file: {'present' if BASELINE_FILE.exists() else 'ABSENT'}; "
      f"adopted by {baseline.get('adopted_by', '-')}, on {baseline.get('on', '-')}")
print(f"cases without a baseline entry: {gaps or 'none'}")
print(f"entries that differ from the proposal (the author chose another execution): {differs or 'none'}")

wrote runs\baseline_proposal_1a872c09.json


baseline file: present; adopted by agent (pipeline rehearsal), on 2026-09-26
cases without a baseline entry: none
entries that differ from the proposal (the author chose another execution): none


## 6. Done when (B10)

The checks the spec names, computed. A `False` is a gap to close before this
run can serve as the baseline.

In [8]:
done = {
    "every case has a baseline entry": not gaps,
    "every entry is structurally clean": all(by_exec.get(e["execution_id"], {}).get("structural_outcome") == "none"
                                             for e in entries.values()),
    "every execution was scored": frame["score"].notna().all(),
    "every version field is set": not missing,
    "the baseline is one run": len(BASELINE_RUN_IDS) == 1,
    "the baseline is the adopted file": not TEST,
}
pd.Series(done, name="holds").to_frame()

,holds
every case has a baseline entry,True
every entry is structurally clean,True
every execution was scored,True
every version field is set,True
the baseline is one run,True
the baseline is the adopted file,False
